In [12]:
import gdown
import pandas as pd
import numpy as np
import yfinance as yf

### Q1

In [13]:
# 1. Data Loading
url = 'https://www.iposcoop.com/ipos-recently-filed/'
tables = pd.read_html(url)
df = tables[0]

# Filter to keep only 'Withdrawn' IPOs
df_withdrawn = df[df['Expected To Trade'] == 'Withdrawn'].copy()

# 2. Company Classification
def classify_company_type(company_name):
    if 'Technologies' in company_name:
        return 'Technologies'
    elif 'Acquisition Corp' in company_name or 'Acquisition Corporation' in company_name or 'Corp' in company_name:
        return 'Acquisition Corp'
    elif 'Inc' in company_name or 'Incorporated' in company_name:
        return 'Inc.'
    elif 'Group' in company_name:
        return 'Group'
    elif 'Ltd' in company_name or 'Limited' in company_name:
        return 'Limited'
    elif 'Holdings' in company_name or 'Holding' in company_name:
        return 'Holdings'
    else:
        return 'Other'

df_withdrawn['Company Type'] = df_withdrawn['Company'].apply(classify_company_type)

# 3. Price Parsing
def parse_price(price_str):
    if pd.isna(price_str) or price_str == '-' or str(price_str).strip() == '':
        return np.nan
    try:
        return float(str(price_str).replace('$', '').replace(',', '').strip())
    except ValueError:
        return np.nan

df_withdrawn['Price Low Numeric'] = df_withdrawn['Price Low'].apply(parse_price)
df_withdrawn['Price High Numeric'] = df_withdrawn['Price High'].apply(parse_price)
df_withdrawn['Avg_price'] = df_withdrawn[['Price Low Numeric', 'Price High Numeric']].mean(axis=1)

# 4. Numeric Conversion
def clean_and_convert_numeric(series):
    return series.astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False).str.strip().replace('-', np.nan).astype(float)

df_withdrawn['Shares (millions)'] = clean_and_convert_numeric(df_withdrawn['Shares (millions)'])
df_withdrawn['Est $ Vol (millions)'] = clean_and_convert_numeric(df_withdrawn['Est $ Vol (millions)'])

# 5. Value Calculation
df_withdrawn['Calculated_Shares_Offered'] = df_withdrawn['Shares (millions)'] * df_withdrawn['Avg_price']
df_withdrawn['Shares_offered_value'] = df_withdrawn['Calculated_Shares_Offered'].fillna(df_withdrawn['Est $ Vol (millions)'])

# 6. Filter by File Date before Sep 11, 2026
df_withdrawn['File Date'] = pd.to_datetime(df_withdrawn['File Date'])
df_filtered_date = df_withdrawn[df_withdrawn['File Date'] < '2026-09-11'].copy()

# 7. Aggregation
final_aggregation = df_filtered_date.groupby('Company Type')['Shares_offered_value'].sum().reset_index()
final_aggregation = final_aggregation.sort_values(by='Shares_offered_value', ascending=False)

display(final_aggregation)

,Company Type,Shares_offered_value
0,Acquisition Corp,499.9850
3,Inc.,351.0000
2,Holdings,311.6575
5,Other,290.4450
4,Limited,197.2500
6,Technologies,184.9000
1,Group,32.5000


### Q2

In [14]:
# 1. Data Loading: Download the list of 2025 IPOs
url_2025 = 'https://www.iposcoop.com/2025-pricings/'
tables_2025 = pd.read_html(url_2025)
df_2025 = tables_2025[0]

# Clean column names
df_2025.columns = [str(col).strip() for col in df_2025.columns]

# Convert 'Offer Date' to datetime
df_2025['Offer Date'] = pd.to_datetime(df_2025['Offer Date'], errors='coerce')

# Clean return column to filter out 0% return
def clean_return(val):
    if pd.isna(val):
        return np.nan
    return float(str(val).replace('%', '').replace('$', '').replace(',', '').strip())

# Often return is represented in a column like 'Return' or 'Current Return'
return_col = [col for col in df_2025.columns if 'Return' in col][0]
df_2025['Cleaned_Return'] = df_2025[return_col].apply(clean_return)

# 2. Filtering: Offer Date before 1 September 2025 and Exclude 0% return
df_filtered_2025 = df_2025[
    (df_2025['Offer Date'] < '2025-09-01') &
    (df_2025['Cleaned_Return'] != 0.0)
].copy()

print(f"Filtered count (should be ~148): {len(df_filtered_2025)}")

# 3. Data Download using yfinance
tickers = df_filtered_2025['Symbol'].dropna().unique().tolist()
print(f"Downloading data for {len(tickers)} tickers...")

# Download historical data from 2025-01-01 to 2026-09-20 to cover 2026-09-11 and calculation windows
stocks_data = yf.download(tickers, start='2025-01-01', end='2026-09-20', group_by='ticker')

# Reshape or iterate to calculate metrics per ticker
rows = []
for ticker in tickers:
    if ticker in stocks_data.columns.levels[0]:
        ticker_df = stocks_data[ticker].dropna(subset=['Close']).copy()
        if len(ticker_df) > 0:
            # Define growth_252d as Close / Close.shift(252)
            ticker_df['growth_252d'] = ticker_df['Close'] / ticker_df['Close'].shift(252)
            # Calculate annualized volatility
            ticker_df['volatility'] = ticker_df['Close'].rolling(30).std() * np.sqrt(252)
            # Sharpe Ratio (Assuming 5% risk free rate)
            ticker_df['Sharpe'] = (ticker_df['growth_252d'] - 0.05) / ticker_df['volatility']
            # Keep the ticker name
            ticker_df['Ticker'] = ticker
            rows.append(ticker_df)

if rows:
    all_stocks_df = pd.concat(rows)
    # Filter to keep data only for '2026-09-11'
    all_stocks_df = all_stocks_df.reset_index()
    all_stocks_df['Date'] = pd.to_datetime(all_stocks_df['Date'])
    target_df = all_stocks_df[all_stocks_df['Date'] == '2026-09-11']

    display(target_df[['Ticker', 'growth_252d', 'volatility', 'Sharpe']].describe())
else:
    print("No stock data downloaded successfully.")

Filtered count (should be ~148): 146


/tmp/ipykernel_6856/1575705605.py:35: FutureWarning: YF.download() has changed argument auto_adjust default to True
  stocks_data = yf.download(tickers, start='2025-01-01', end='2026-09-20', group_by='ticker')
[*********************100%***********************]  146 of 146 completed
ERROR:yfinance:
15 Failed downloads:
ERROR:yfinance:['EMPG', 'CAEP', 'PTNM', 'AHL', 'AGH', 'SDM', 'MJID', 'MCTR', 'EPWK', 'TBH', 'WGRX', 'CEPT', 'MTSR', 'SKBL']: YFTzMissingError('possibly delisted; no timezone found')
ERROR:yfinance:['DLXY']: YFPricesMissingError('possibly delisted; no price data found  (1d 2025-01-01 -> 2026-09-20)')


Price,growth_252d,volatility,Sharpe
count,129.000000,130.000000,129.000000
mean,1.065215,27.725051,inf
std,3.059657,67.095064,NaN
min,0.001005,0.000000,-0.040147
25%,0.149038,2.325895,0.011942
50%,0.596026,7.927902,0.048009
75%,1.040233,23.659042,0.134248
max,33.638270,627.698106,inf


In [15]:
# Calculate and print specific metrics to answer the user's questions
if 'target_df' in globals() and not target_df.empty:
    median_sharpe = target_df['Sharpe'].median()
    mean_sharpe = target_df['Sharpe'].mean()
    skewness_growth = target_df['growth_252d'].skew()
    best_performer = target_df.loc[target_df['Sharpe'].idxmax()]

    print(f"--- Metrics as of September 11, 2026 ---")
    print(f"Median Sharpe Ratio: {median_sharpe:.4f}")
    print(f"Mean Sharpe Ratio: {mean_sharpe:.4f}")
    print(f"Skewness of 1-Year Growth: {skewness_growth:.4f}")
    print(f"Top Sharpe Ratio Performer: {best_performer['Ticker']} (Sharpe: {best_performer['Sharpe']:.4f}, Volatility: {best_performer['volatility']:.4f})")

    # Display standard descriptive statistics table
    display(target_df[['growth_252d', 'volatility', 'Sharpe']].describe())
else:
    print("Target DataFrame is empty or not found in the environment.")

--- Metrics as of September 11, 2026 ---
Median Sharpe Ratio: 0.0480
Mean Sharpe Ratio: inf
Skewness of 1-Year Growth: 9.6690
Top Sharpe Ratio Performer: MAGH (Sharpe: inf, Volatility: 0.0000)


Price,growth_252d,volatility,Sharpe
count,129.000000,130.000000,129.000000
mean,1.065215,27.725051,inf
std,3.059657,67.095064,NaN
min,0.001005,0.000000,-0.040147
25%,0.149038,2.325895,0.011942
50%,0.596026,7.927902,0.048009
75%,1.040233,23.659042,0.134248
max,33.638270,627.698106,inf


In [16]:
# Clean up zero-volatility entries causing infinite Sharpe ratios
target_df_cleaned = target_df[target_df['volatility'] > 0.0].copy()

# Recalculate metrics
median_sharpe_clean = target_df_cleaned['Sharpe'].median()
mean_sharpe_clean = target_df_cleaned['Sharpe'].mean()
skewness_growth_clean = target_df_cleaned['growth_252d'].skew()
best_performer_clean = target_df_cleaned.loc[target_df_cleaned['Sharpe'].idxmax()]

print("--- Cleaned Metrics (Excluding Volatility = 0) as of September 11, 2026 ---")
print(f"Median Sharpe Ratio: {median_sharpe_clean:.4f}")
print(f"Mean Sharpe Ratio: {mean_sharpe_clean:.4f}")
print(f"Skewness of 1-Year Growth: {skewness_growth_clean:.4f}")
print(f"Top Sharpe Ratio Performer: {best_performer_clean['Ticker']} (Sharpe: {best_performer_clean['Sharpe']:.4f}, Volatility: {best_performer_clean['volatility']:.4f})")

# Display cleaned descriptive statistics
display(target_df_cleaned[['growth_252d', 'volatility', 'Sharpe']].describe())

--- Cleaned Metrics (Excluding Volatility = 0) as of September 11, 2026 ---
Median Sharpe Ratio: 0.0459
Mean Sharpe Ratio: 0.1867
Skewness of 1-Year Growth: 10.2194
Top Sharpe Ratio Performer: HCMAU (Sharpe: 5.3853, Volatility: 0.1828)


Price,growth_252d,volatility,Sharpe
count,126.000000,126.000000,126.000000
mean,0.978745,28.605211,0.186729
std,3.029339,67.973708,0.580669
min,0.001005,0.182827,-0.040147
25%,0.141035,2.513443,0.010027
50%,0.584982,8.388675,0.045858
75%,1.029428,24.096718,0.121250
max,33.638270,627.698106,5.385326


### Q3

In [17]:
# We will use all_stocks_df containing historical price data for the 2025 IPOs
if 'all_stocks_df' in globals() and not all_stocks_df.empty:
    # 1. Identify Entry Points (first available trading day and close price for each ticker)
    all_stocks_df['Date'] = pd.to_datetime(all_stocks_df['Date'])

    # Group by Ticker to find the absolute first date (IPO/listing date)
    entry_records = all_stocks_df.groupby('Ticker').agg(
        First_Date=('Date', 'min')
    ).reset_index()

    # Get the closing price on that first date
    entry_prices = pd.merge(
        entry_records,
        all_stocks_df[['Ticker', 'Date', 'Close']],
        left_on=['Ticker', 'First_Date'],
        right_on=['Ticker', 'Date']
    ).rename(columns={'Close': 'First_Close'}).drop(columns=['Date'])

    # 2. Calculate holding periods (21 to 252 trading days)
    # We need to compute holding period growth rates relative to 'First_Close'.
    # An elegant way is to sort by Ticker and Date, then use shifts or indexing.
    holding_results = []

    for ticker, group in all_stocks_df.sort_values(['Ticker', 'Date']).groupby('Ticker'):
        if ticker in entry_prices['Ticker'].values:
            first_close = entry_prices.loc[entry_prices['Ticker'] == ticker, 'First_Close'].values[0]

            # Create dictionary to store results for this ticker
            ticker_growth = {'Ticker': ticker, 'First_Date': group['Date'].min(), 'First_Close': first_close}

            # Calculate growth for 1 to 12 months (21 to 252 trading days)
            # index 0 is first day. Month 1 is index 21, Month 2 is index 42, etc.
            group_reset = group.reset_index(drop=True)
            for m in range(1, 13):
                trading_days = m * 21
                if trading_days < len(group_reset):
                    close_at_m = group_reset.loc[trading_days, 'Close']
                    # Growth = Close_at_m / First_Close
                    ticker_growth[f'future_growth_{m}_m'] = close_at_m / first_close
                else:
                    ticker_growth[f'future_growth_{m}_m'] = np.nan

            holding_results.append(ticker_growth)

    growth_df = pd.DataFrame(holding_results)

    # 3. Descriptive Statistics
    growth_cols = [f'future_growth_{m}_m' for m in range(1, 13)]
    stats_df = growth_df[growth_cols].describe().T
    stats_df['Month'] = range(1, 13)
    stats_df = stats_df[['Month', 'count', 'mean', '50%', 'min', 'max']]
    stats_df.rename(columns={'50%': 'median'}, inplace=True)

    display(stats_df)

    # Print optimal month
    optimal_row = stats_df.loc[stats_df['median'].idxmax()]
    print(f"\nOptimal holding period is {int(optimal_row['Month'])} month(s) with a median growth of {optimal_row['median']:.4f}.")
else:
    print("all_stocks_df not found or empty.")

,Month,count,mean,median,min,max
future_growth_1_m,1,131.0,96.475452,0.930622,0.072941,12500.000279
future_growth_2_m,2,130.0,68.494523,0.891684,0.106197,8750.000196
future_growth_3_m,3,130.0,51.642359,0.827951,0.090147,6562.500147
future_growth_4_m,4,130.0,22.872042,0.736039,0.063927,2837.500063
future_growth_5_m,5,130.0,54.660308,0.695853,0.032245,6975.000156
future_growth_6_m,6,130.0,64.504145,0.733385,0.022075,8250.000184
future_growth_7_m,7,130.0,59.722684,0.667972,0.013229,7625.000170
future_growth_8_m,8,130.0,21.624139,0.603984,0.015130,2687.500060
future_growth_9_m,9,130.0,20.163352,0.584421,0.014675,2475.000055
future_growth_10_m,10,130.0,11.346041,0.533487,0.011947,1325.000030



Optimal holding period is 1 month(s) with a median growth of 0.9306.


### Q4

In [18]:
# 1. Download the dataset
file_id = "1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-"
gdown.download(f"https://drive.google.com/uc?id={file_id}", "data.parquet", quiet=False)

# 2. Load the dataset
df_rsi = pd.read_parquet("data.parquet", engine="pyarrow")
display(df_rsi.head())

Downloading...
From (original): https://drive.google.com/uc?id=1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-
From (redirected): https://drive.google.com/uc?id=1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-&confirm=t&uuid=fd9243ce-97ac-4d72-8d5c-35571e9a4d11
To: /content/data.parquet
100%|██████████| 130M/130M [00:00<00:00, 148MB/s]


,Open,High,Low,Close_x,Volume,Dividends,Stock Splits,Ticker,Year,Month,...,growth_brent_oil_7d,growth_brent_oil_30d,growth_brent_oil_90d,growth_brent_oil_365d,growth_btc_usd_1d,growth_btc_usd_3d,growth_btc_usd_7d,growth_btc_usd_30d,growth_btc_usd_90d,growth_btc_usd_365d
0,0.054277,0.062259,0.054277,0.059598,1.031789e+09,0.0,0.0,MSFT,1986,1986-03-01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.059598,0.062791,0.059598,0.061726,3.081600e+08,0.0,0.0,MSFT,1986,1986-03-01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.061726,0.063323,0.061726,0.062791,1.331712e+08,0.0,0.0,MSFT,1986,1986-03-01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0.062791,0.063323,0.060662,0.061194,6.776640e+07,0.0,0.0,MSFT,1986,1986-03-01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0.061194,0.061726,0.059598,0.060130,4.789440e+07,0.0,0.0,MSFT,1986,1986-03-01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
# 3. Strategy Setup and Filtering
# Convert Date index or column to datetime if not already
if 'Date' in df_rsi.columns:
    df_rsi['Date'] = pd.to_datetime(df_rsi['Date'])
    df_filtered_rsi = df_rsi[(df_rsi['Date'] >= '2000-01-01') & (df_rsi['Date'] <= '2025-06-01')].copy()
else:
    df_rsi.index = pd.to_datetime(df_rsi.index)
    df_filtered_rsi = df_rsi.loc['2000-01-01':'2025-06-01'].copy()

# Filter for oversold signals where RSI < 30
# Let's inspect column names first to find the exact column name for RSI and growth_future_30d
print("Columns:", df_filtered_rsi.columns.tolist())

Columns: ['Open', 'High', 'Low', 'Close_x', 'Volume', 'Dividends', 'Stock Splits', 'Ticker', 'Year', 'Month', 'Weekday', 'Date', 'growth_1d', 'growth_3d', 'growth_7d', 'growth_30d', 'growth_90d', 'growth_365d', 'growth_future_30d', 'SMA10', 'SMA20', 'growing_moving_average', 'high_minus_low_relative', 'volatility', 'is_positive_growth_30d_future', 'ticker_type', 'index_x', 'adx', 'adxr', 'apo', 'aroon_1', 'aroon_2', 'aroonosc', 'bop', 'cci', 'cmo', 'dx', 'macd', 'macdsignal', 'macdhist', 'macd_ext', 'macdsignal_ext', 'macdhist_ext', 'macd_fix', 'macdsignal_fix', 'macdhist_fix', 'mfi', 'minus_di', 'mom', 'plus_di', 'dm', 'ppo', 'roc', 'rocp', 'rocr', 'rocr100', 'rsi', 'slowk', 'slowd', 'fastk', 'fastd', 'fastk_rsi', 'fastd_rsi', 'trix', 'ultosc', 'willr', 'index_y', 'ad', 'adosc', 'obv', 'atr', 'natr', 'ht_dcperiod', 'ht_dcphase', 'ht_phasor_inphase', 'ht_phasor_quadrature', 'ht_sine_sine', 'ht_sine_leadsine', 'ht_trendmod', 'avgprice', 'medprice', 'typprice', 'wclprice', 'index', 'cdl2

In [20]:
# Assuming standard column names are 'rsi' and 'growth_future_30d'
rsi_col = [col for col in df_filtered_rsi.columns if 'rsi' in col.lower()][0]
growth_col = [col for col in df_filtered_rsi.columns if 'growth_future_30d' in col.lower()][0]

oversold_trades = df_filtered_rsi[df_filtered_rsi[rsi_col] < 30].copy()

# 4. Profit Calculation
# Net income = sum of (1000 * (growth_future_30d - 1)) for each trade
net_income = 1000 * (oversold_trades[growth_col] - 1).sum()
profit_thousands = net_income / 1000.0

print(f"Number of trades triggered: {len(oversold_trades)}")
print(f"Total Profit (in $ thousands): ${profit_thousands:.3f}k")

Number of trades triggered: 5206
Total Profit (in $ thousands): $65.806k
